In [1]:
%pip install -U massive pandas python-dateutil

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: massive in c:\users\amrit\anaconda3\lib\site-packages (2.2.0)
   ---------------------------------------- 0.0/65.4 kB ? eta -:--:--
   ------------------------------------- -- 61.4/65.4 kB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 65.4/65.4 kB 586.9 kB/s eta 0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 10.2 MB/s eta 0:00:01
   --- ------------------------------------ 0.9/9.7 MB 9.3 MB/s eta 0:00:01
   ----- ---------------------------------- 1.3/9.7 MB 8.3 MB/s eta 0:00:02
   ------ --------------------------------- 1.6/9.7 MB 5.9 MB/s eta 0:00:02
   -------- ------------------------------- 2.0/9.7 MB 5.0 MB/s eta 0:00:02
   ---------- ----------------------------- 2.5/9.7 MB 5.4 MB/s eta 0:00:02
   ------------ --------------------------- 2.9/9.7 MB 5.7 MB/s eta 0:00:02
   ---------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
recordlinkage 0.0.0 requires pandas<3,>=1, but you have pandas 3.0.1 which is incompatible.
streamlit 1.32.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.1 which is incompatible.


In [18]:
from getpass import getpass
from datetime import date 
import time
from dateutil.relativedelta import relativedelta
import pandas as pd
from massive import RESTClient

In [11]:
# # ---------- ENTER API KEY SAFELY ----------
# API_KEY = getpass("Enter your Massive API key: ")  # <-- type key here (input hidden)
# # -----------------------------------------

In [13]:
API_KEY = "AXWlAis8pJmpQEQp_Lsh4ykpR33zlQuO".strip()
print("Key length:", len(API_KEY))

Key length: 32


In [14]:
TICKERS = [
    "NOC", "RTX", "JNJ", "LMT", "NSRGF", "DUK", "CNI",
    "RIVN", "PLUG", "COIN", "U", "PM", "XOM", "PLTR", "META", "GLNCY"
]

In [15]:
client = RESTClient(api_key=API_KEY)

end = date.today()
start = end - relativedelta(years=3)

In [16]:
def fetch_daily_closes(ticker, start_date, end_date, max_tries=8):
    for attempt in range(max_tries):
        try:
            rows = []
            for bar in client.list_aggs(
                ticker=ticker,
                multiplier=1,
                timespan="day",
                from_=start_date.isoformat(),
                to=end_date.isoformat(),
                limit=50000,
            ):
                rows.append((pd.to_datetime(bar.timestamp, unit="ms"), float(bar.close)))

            if not rows:
                raise ValueError(f"No data returned for {ticker}")

            return (
                pd.Series(
                    data=[c for _, c in rows],
                    index=[ts for ts, _ in rows],
                    name=ticker,
                )
                .sort_index()
            )

        except Exception as e:
            if "429" in str(e):
                sleep_s = min(60, 5 * (2 ** attempt))
                print(f"429 for {ticker}. Sleeping {sleep_s}s then retrying...")
                time.sleep(sleep_s)
                continue
            raise

    raise RuntimeError(f"Failed for {ticker} after {max_tries} retries due to repeated 429s.")


In [19]:
price_series = {}

for t in TICKERS:
    print(f"Fetching {t}...")
    price_series[t] = fetch_daily_closes(t, start, end)
    time.sleep(3)  # spacing between tickers

prices = pd.DataFrame(price_series)
prices.head()


Fetching NOC...
Fetching RTX...
Fetching JNJ...
Fetching LMT...
Fetching NSRGF...
Fetching DUK...
429 for DUK. Sleeping 5s then retrying...
Fetching CNI...
Fetching RIVN...
Fetching PLUG...
Fetching COIN...
Fetching U...
429 for U. Sleeping 5s then retrying...
429 for U. Sleeping 10s then retrying...
429 for U. Sleeping 20s then retrying...
429 for U. Sleeping 40s then retrying...
Fetching PM...
Fetching XOM...
Fetching PLTR...
Fetching META...
Fetching GLNCY...
429 for GLNCY. Sleeping 5s then retrying...


,NOC,RTX,JNJ,LMT,NSRGF,DUK,CNI,RIVN,PLUG,COIN,U,PM,XOM,PLTR,META,GLNCY
2024-03-20 04:00:00,470.75,94.85,155.76,440.41,106.870,95.41,131.88,11.36,3.52,256.88,27.15,93.64,112.99,24.57,505.52,10.740
2024-03-21 04:00:00,467.49,94.26,155.75,443.16,105.800,94.96,132.83,11.17,3.59,262.00,27.57,92.20,113.49,24.49,507.76,10.797
2024-03-22 04:00:00,468.75,95.54,155.23,445.88,104.936,94.61,131.83,10.80,3.41,255.51,26.99,90.88,113.49,24.18,509.58,10.730
2024-03-25 04:00:00,469.32,95.63,155.22,446.31,105.194,94.84,129.78,10.65,3.33,279.71,27.20,91.15,114.65,24.51,503.02,10.600
2024-03-26 04:00:00,469.91,96.06,155.77,445.99,105.154,93.67,130.17,10.52,3.22,266.81,26.70,90.38,113.79,24.89,495.89,10.630


In [22]:
# returns = prices.pct_change().dropna(how="all")

# daily_var = returns.var()
# ann_vol = returns.std() * (252 ** 0.5)

# risk_bucket = pd.qcut(
#     ann_vol.rank(method="first"),
#     q=3,
#     labels=["Safe", "Moderate", "Volatile"]
# )

# risk_table = (
#     pd.DataFrame({
#         "ticker": ann_vol.index,
#         "daily_return_variance": daily_var.values,
#         "annualized_volatility": ann_vol.values,
#         "risk_category": risk_bucket.values,
#     })
#     .sort_values("annualized_volatility")
#     .reset_index(drop=True)
# )

# risk_table

In [23]:
# risk_table.to_csv("risk_profiles_3y_quantiles.csv", index=False)

In [24]:
# --------------------------------------------------
# 1) Clean index so date matching works properly
# --------------------------------------------------
prices.index = pd.to_datetime(prices.index).normalize()
prices = prices.sort_index()

# --------------------------------------------------
# 2) Compute returns and BINARY risk categories only
# --------------------------------------------------
returns = prices.pct_change().dropna(how="all")

daily_var = returns.var()
ann_vol = returns.std() * (252 ** 0.5)

# 2 groups only: Safe vs Risky
risk_bucket = pd.qcut(
    ann_vol.rank(method="first"),
    q=2,
    labels=["Safe", "Risky"]
)

risk_table = (
    pd.DataFrame({
        "ticker": ann_vol.index,
        "daily_return_variance": daily_var.values,
        "annualized_volatility": ann_vol.values,
        "risk_category": risk_bucket.values
    })
    .sort_values("annualized_volatility")
    .reset_index(drop=True)
)

print("BINARY RISK TABLE")
display(risk_table)

# --------------------------------------------------
# 3) Helper: get latest available close on or before target date
# --------------------------------------------------
def get_last_valid_price(series, target_date):
    target_date = pd.Timestamp(target_date).normalize()
    s = series.dropna().sort_index()
    s = s[s.index <= target_date]
    
    if s.empty:
        return pd.NaT, np.nan
    
    return s.index[-1], float(s.iloc[-1])

# --------------------------------------------------
# 4) Set your investment window
#    Change the YEAR here if needed
# --------------------------------------------------
entry_date = "2026-02-19"
exit_date  = "2026-03-19"

# Example: assume investor puts the same amount into each stock
investment_amount = 100.0

# --------------------------------------------------
# 5) Compute what that investment becomes after 1 month
# --------------------------------------------------
investment_rows = []

for ticker in TICKERS:
    entry_used, entry_price = get_last_valid_price(prices[ticker], entry_date)
    exit_used, exit_price = get_last_valid_price(prices[ticker], exit_date)

    if pd.isna(entry_price) or pd.isna(exit_price):
        continue

    shares_bought = investment_amount / entry_price
    final_value = shares_bought * exit_price
    profit_loss = final_value - investment_amount
    one_month_return = profit_loss / investment_amount

    investment_rows.append({
        "ticker": ticker,
        "entry_date_used": entry_used.date(),
        "entry_price": entry_price,
        "exit_date_used": exit_used.date(),
        "exit_price": exit_price,
        "investment_amount": investment_amount,
        "shares_bought": shares_bought,
        "final_value": final_value,
        "profit_loss": profit_loss,
        "one_month_return": one_month_return,
        "outcome": "Profit" if profit_loss > 0 else "Loss"
    })

investment_table = pd.DataFrame(investment_rows)

print("1-MONTH INVESTMENT RESULTS")
display(investment_table.sort_values("one_month_return"))

# --------------------------------------------------
# 6) Merge risk + 1-month outcome into one final table
# --------------------------------------------------
final_table = (
    risk_table.merge(investment_table, on="ticker", how="left")
    .sort_values(["risk_category", "annualized_volatility"])
    .reset_index(drop=True)
)

print("FINAL TABLE: RISK + 1-MONTH OUTCOME")
display(final_table)

# --------------------------------------------------
# 7) Quick summary checks
# --------------------------------------------------
print("\nCounts by risk category:")
print(final_table["risk_category"].value_counts())

print("\nCounts by outcome:")
print(final_table["outcome"].value_counts())

print("\nRisk x Outcome table:")
print(pd.crosstab(final_table["risk_category"], final_table["outcome"]))

# Optional: save
final_table.to_csv("risk_and_1m_outcomes.csv", index=False)

BINARY RISK TABLE


,ticker,daily_return_variance,annualized_volatility,risk_category
0,DUK,0.000107,0.164318,Safe
1,JNJ,0.000129,0.180021,Safe
2,CNI,0.000186,0.216584,Safe
3,XOM,0.000203,0.226035,Safe
4,RTX,0.000230,0.240592,Safe
5,LMT,0.000231,0.241473,Safe
6,NSRGF,0.000246,0.248879,Safe
7,NOC,0.000246,0.248980,Safe
8,PM,0.000246,0.249135,Risky
9,GLNCY,0.000478,0.347157,Risky


1-MONTH INVESTMENT RESULTS


,ticker,entry_date_used,entry_price,exit_date_used,exit_price,investment_amount,shares_bought,final_value,profit_loss,one_month_return,outcome
11,PM,2026-02-19,183.50,2026-03-19,163.370,100.0,0.544959,89.029973,-10.970027,-0.109700,Loss
6,CNI,2026-02-19,109.43,2026-03-19,99.100,100.0,0.913826,90.560175,-9.439825,-0.094398,Loss
4,NSRGF,2026-02-19,105.55,2026-03-19,96.400,100.0,0.947418,91.331123,-8.668877,-0.086689,Loss
14,META,2026-02-19,644.78,2026-03-19,606.700,100.0,0.155092,94.094110,-5.905890,-0.059059,Loss
3,LMT,2026-02-19,666.51,2026-03-19,637.510,100.0,0.150035,95.648978,-4.351022,-0.043510,Loss
2,JNJ,2026-02-19,246.91,2026-03-19,237.600,100.0,0.405006,96.229395,-3.770605,-0.037706,Loss
0,NOC,2026-02-19,736.87,2026-03-19,714.150,100.0,0.135709,96.916688,-3.083312,-0.030833,Loss
1,RTX,2026-02-19,205.41,2026-03-19,200.730,100.0,0.486831,97.721630,-2.278370,-0.022784,Loss
15,GLNCY,2026-02-19,13.65,2026-03-19,13.875,100.0,7.326007,101.648352,1.648352,0.016484,Profit
5,DUK,2026-02-19,126.37,2026-03-19,129.740,100.0,0.791327,102.666772,2.666772,0.026668,Profit


FINAL TABLE: RISK + 1-MONTH OUTCOME


,ticker,daily_return_variance,annualized_volatility,risk_category,entry_date_used,entry_price,exit_date_used,exit_price,investment_amount,shares_bought,final_value,profit_loss,one_month_return,outcome
0,DUK,0.000107,0.164318,Safe,2026-02-19,126.37,2026-03-19,129.740,100.0,0.791327,102.666772,2.666772,0.026668,Profit
1,JNJ,0.000129,0.180021,Safe,2026-02-19,246.91,2026-03-19,237.600,100.0,0.405006,96.229395,-3.770605,-0.037706,Loss
2,CNI,0.000186,0.216584,Safe,2026-02-19,109.43,2026-03-19,99.100,100.0,0.913826,90.560175,-9.439825,-0.094398,Loss
3,XOM,0.000203,0.226035,Safe,2026-02-19,150.97,2026-03-19,158.160,100.0,0.662383,104.762536,4.762536,0.047625,Profit
4,RTX,0.000230,0.240592,Safe,2026-02-19,205.41,2026-03-19,200.730,100.0,0.486831,97.721630,-2.278370,-0.022784,Loss
5,LMT,0.000231,0.241473,Safe,2026-02-19,666.51,2026-03-19,637.510,100.0,0.150035,95.648978,-4.351022,-0.043510,Loss
6,NSRGF,0.000246,0.248879,Safe,2026-02-19,105.55,2026-03-19,96.400,100.0,0.947418,91.331123,-8.668877,-0.086689,Loss
7,NOC,0.000246,0.248980,Safe,2026-02-19,736.87,2026-03-19,714.150,100.0,0.135709,96.916688,-3.083312,-0.030833,Loss
8,PM,0.000246,0.249135,Risky,2026-02-19,183.50,2026-03-19,163.370,100.0,0.544959,89.029973,-10.970027,-0.109700,Loss
9,GLNCY,0.000478,0.347157,Risky,2026-02-19,13.65,2026-03-19,13.875,100.0,7.326007,101.648352,1.648352,0.016484,Profit



Counts by risk category:
risk_category
Safe     8
Risky    8
Name: count, dtype: int64

Counts by outcome:
outcome
Profit    8
Loss      8
Name: count, dtype: int64

Risk x Outcome table:
outcome        Loss  Profit
risk_category              
Safe              6       2
Risky             2       6


In [28]:
# --------------------------------------------------
# 8) Add CONTEXT columns 
# --------------------------------------------------

# Risk window (based on your earlier code)
risk_start_date = start
risk_end_date = end

# Add readable metadata columns (row-level clarity)
final_table["risk_window"] = f"{risk_start_date} to {risk_end_date} (3Y daily volatility)"
final_table["investment_window_requested"] = f"{entry_date} to {exit_date} (1M horizon)"

# Rename for clarity
final_table = final_table.rename(columns={
    "entry_date_used": "actual_entry_date_used",
    "exit_date_used": "actual_exit_date_used"
})

# Reorder columns for readability
final_table = final_table[
    [
        "ticker",
        "risk_category",
        "annualized_volatility",
        "daily_return_variance",
        
        "risk_window",
        
        "investment_window_requested",
        "actual_entry_date_used",
        "entry_price",
        "actual_exit_date_used",
        "exit_price",
        
        "investment_amount",
        "shares_bought",
        "final_value",
        "profit_loss",
        "one_month_return",
        "outcome"
    ]
]

# --------------------------------------------------
# 9) SAVE CLEAN DATASET 
# --------------------------------------------------
final_table.to_csv("market_mindset_final_dataset.csv", index=False)
print("Saved: market_mindset_final_dataset.csv")


# --------------------------------------------------
# 10) ADD METADATA ROW 
# --------------------------------------------------

# Create metadata row with SAME columns
metadata_row = {col: "" for col in final_table.columns}

metadata_row["ticker"] = "METADATA"
metadata_row["risk_category"] = f"Risk window: {risk_start_date} to {risk_end_date} (3 years)"
metadata_row["annualized_volatility"] = f"Investment window: {entry_date} to {exit_date}"
metadata_row["daily_return_variance"] = "Volatility = std(returns) * sqrt(252)"

metadata_df = pd.DataFrame([metadata_row])

# Combine metadata + actual data
final_with_metadata = pd.concat([metadata_df, final_table], ignore_index=True)

# Save
final_with_metadata.to_csv("market_mindset_final_dataset_with_metadata.csv", index=False)

print("Saved: market_mindset_final_dataset_with_metadata.csv")

Saved: market_mindset_final_dataset.csv
Saved: market_mindset_final_dataset_with_metadata.csv


In [1]:
final_table[["ticker", "risk_category", "outcome", "one_month_return"]]

NameError: name 'final_table' is not defined